# Multi-year training (2021-2024), test on 2025 and 2026, sharper borders

Train on 2021-2024 in the 21 training regions. Validate on the 6 validation regions in 2024. Test on 2025 and 2026 in regions
the model never trained on (test + test_ood). So the test is held out in **space and in time**.

Two models are trained on the same data, 3 seeds each:
- **V12** (same recipe as the shipped model, now with 4 years)
- **V16** (V12 plus extra loss weight on class borders, for sharper maps)

The shipped 2021-only V12 is scored on the same 2025 and 2026 data. That comparison decides what ships.

Limits to put in the report: 2025 and 2026 only have Dynamic World as reference, the same label family the model is trained on.
There is no independent map for those years here. 2026 has Jan-Oct only. The 2026 yearly statistics use 10 months, not 12.

In [1]:
# 0. Paths + pristine config guard (this config now has the years: block)
import os, shutil, yaml
BASE = globals().get('BASE', '/home/jovyan/vault/lulc'); OUT = globals().get('OUT', f'{BASE}/raw')
RUNS = globals().get('RUNS', f'{BASE}/runs'); PROJECT = globals().get('PROJECT', 'sar-lulc')
os.chdir('/home/jovyan/vault/sar-lulc')
RUNS_MY = f'{BASE}/runs_multiyear'; os.makedirs(RUNS_MY, exist_ok=True)
for v in ['V0_scratch_1m','V1_imagenet_1m']:      # the encoder-weights decision reads V0/V1 scores
    if os.path.isdir(f'{RUNS}/{v}') and not os.path.isdir(f'{RUNS_MY}/{v}'): shutil.copytree(f'{RUNS}/{v}', f'{RUNS_MY}/{v}')
cfg = yaml.safe_load(open('configs/regions.yaml')); R = cfg['regions']
keys = lambda *sp: ' '.join(k for k, v in R.items() if v['split'] in sp)
TRAIN, VAL, TEST = keys('train'), keys('val'), keys('test', 'test_ood')
print('years block:', cfg.get('years'))
print(len(TRAIN.split()), 'train |', len(VAL.split()), 'val |', len(TEST.split()), 'test regions')

years block: {'train': [2021, 2022, 2023, 2024], 'val': [2024], 'test': [2025, 2026]}
21 train | 6 val | 39 test regions


In [2]:
!pip -q install -r requirements.txt 2>&1 | tail -1
import ee
try: ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook'); ee.Initialize(project=PROJECT)

[notice] To update, run: python3 -m pip install --upgrade pip


/home/jovyan/.local/lib/python3.11/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.74.0. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


## 1. Check the data exists for each year (a few minutes)
Read `s1_median_per_month` per year. Sentinel-1B stopped in Dec 2021, so 2022-2024 have fewer scenes per month than 2021 in many places,
and Sentinel-1C adds scenes again from 2025. A model trained only on 2021 has never seen that. If 2025 or 2026 show 0-1 scenes
per month, say so in the report.

In [3]:
SAMPLE = 'munich berlin paris madrid poltava black_forest kenya_savanna mongolia_steppe prague frankfurt iowa_maize sahel_niger'
%run scripts/ee_year_stats.py --project $PROJECT --years 2021 2022 2023 2024 2025 2026 --regions $SAMPLE

[1/72] munich 2021 done
[2/72] munich 2022 done
[3/72] munich 2023 done
[4/72] munich 2024 done
[5/72] munich 2025 done
[6/72] munich 2026 done
[7/72] berlin 2021 done
[8/72] berlin 2022 done
[9/72] berlin 2023 done
[10/72] berlin 2024 done
[11/72] berlin 2025 done
[12/72] berlin 2026 done
[13/72] paris 2021 done
[14/72] paris 2022 done
[15/72] paris 2023 done
[16/72] paris 2024 done
[17/72] paris 2025 done
[18/72] paris 2026 done
[19/72] madrid 2021 done
[20/72] madrid 2022 done
[21/72] madrid 2023 done
[22/72] madrid 2024 done
[23/72] madrid 2025 done
[24/72] madrid 2026 done
[25/72] prague 2021 done
[26/72] prague 2022 done
[27/72] prague 2023 done
[28/72] prague 2024 done
[29/72] prague 2025 done
[30/72] prague 2026 done
[31/72] frankfurt 2021 done
[32/72] frankfurt 2022 done
[33/72] frankfurt 2023 done
[34/72] frankfurt 2024 done
[35/72] frankfurt 2025 done
[36/72] frankfurt 2026 done
[37/72] poltava 2021 done
[38/72] poltava 2022 done
[39/72] poltava 2023 done
[40/72] poltava 202

## 2. Estimate the export time first (dry run, no download)

In [4]:
%run scripts/export_dataset.py --project $PROJECT --out $OUT --year 2022 --regions $TRAIN --dry-run

munich     train      EPSG:32632 origin=(680090,5346550) 2400px, 12/12 months to fetch
berlin     train      EPSG:32633 origin=(379770,5832080) 2400px, 12/12 months to fetch
hamburg    train      EPSG:32632 origin=(553830,5946040) 2400px, 12/12 months to fetch
cologne    train      EPSG:32632 origin=(344680,5656860) 2400px, 12/12 months to fetch
paris      train      EPSG:32631 origin=(440480,5423720) 2400px, 12/12 months to fetch
amsterdam  train      EPSG:32631 origin=(617630,5815640) 2400px, 12/12 months to fetch
warsaw     train      EPSG:32634 origin=(488830,5798590) 2400px, 12/12 months to fetch
rome       train      EPSG:32633 origin=(280330,4654020) 2400px, 12/12 months to fetch
madrid     train      EPSG:32630 origin=(428290,4486260) 2400px, 12/12 months to fetch
vienna     train      EPSG:32633 origin=(590060,5352360) 2400px, 12/12 months to fetch
poltava    train      EPSG:32636 origin=(531230,5466050) 1024px, 12/12 months to fetch
hungarian_plain train      EPSG:32634 origi

## 3. Export (resumable; re-run the cell if the session stops)
Order matters if time is short: training years first, then 2025 test, then 2026 test.
Training: 3 extra years x 21 regions. Validation: 2024. Test: 2025 and 2026 (2026 = months 1-10).

In [ ]:
for y in (2022, 2023, 2024):
    print('=== training year', y)
    %run scripts/export_dataset.py --project $PROJECT --out $OUT --year $y --regions $TRAIN

=== training year 2022
munich: grid ok (max pixel-centre deviation 1.9e-06 deg), orbit DESCENDING
  2022-01 13s, left ~57 min | {"s1_valid": 1.0, "label_valid": 1.0, "both_valid": 1.0, "s1_scenes_median": 5.0, "label_conf_mean": 48.591}
  2022-02 12s, left ~53 min | {"s1_valid": 1.0, "label_valid": 1.0, "both_valid": 1.0, "s1_scenes_median": 5.0, "label_conf_mean": 53.1}
  2022-03 19s, left ~61 min | {"s1_valid": 1.0, "label_valid": 1.0, "both_valid": 1.0, "s1_scenes_median": 5.0, "label_conf_mean": 57.158}
  2022-04 11s, left ~58 min | {"s1_valid": 1.0, "label_valid": 1.0, "both_valid": 1.0, "s1_scenes_median": 4.0, "label_conf_mean": 58.594}


In [ ]:
print('=== validation year 2024')
%run scripts/export_dataset.py --project $PROJECT --out $OUT --year 2024 --regions $VAL

In [ ]:
print('=== test year 2025')
%run scripts/export_dataset.py --project $PROJECT --out $OUT --year 2025 --regions $TEST

In [ ]:
print('=== test year 2026 (Jan-Oct)')
%run scripts/export_dataset.py --project $PROJECT --out $OUT --year 2026 --months 1 2 3 4 5 6 7 8 9 10 --regions $TEST

## 4. Train V12 and V16 on 2021-2024 (3 seeds each)
`--workers 0` avoids the shared-memory crash. Memory use is about 4x the one-year run. If the kernel dies, train on
`--train-years 2021 2023 2024` instead and say so in the report.

In [ ]:
%run scripts/train_all.py --data $OUT --out $RUNS_MY --variants V12_window3_balanced_stats V16_window3_boundary --seeds 0 1 2 --workers 0

## 5. Test on 2025 and 2026 (held-out regions, future years)
Three models on the same data: the shipped 2021-only V12, V12 on 4 years, V16 on 4 years.
Read: `dw` mIoU (all, cities, rural), per-class IoU, **Boundary F1**, and the predicted/reference border ratio.

In [ ]:
for y in (2025, 2026):
    print(f'################ {y}: shipped V12 (2021 only)')
    %run scripts/evaluate.py --data $OUT --runs $RUNS --variant V12_window3_balanced_stats --year $y --tag _y{y}_shipped
    print(f'################ {y}: V12 trained 2021-2024')
    %run scripts/evaluate.py --data $OUT --runs $RUNS_MY --variant V12_window3_balanced_stats --year $y --tag _y{y}
    print(f'################ {y}: V16 trained 2021-2024 (boundary loss)')
    %run scripts/evaluate.py --data $OUT --runs $RUNS_MY --variant V16_window3_boundary --year $y --tag _y{y}

In [ ]:
# Side-by-side summary. Rows = model, columns = test year.
import pandas as pd, glob, json
rows=[]
for y in (2025, 2026):
    for name, runs, var, tag in [('shipped 2021-only', RUNS, 'V12_window3_balanced_stats', f'_y{y}_shipped'),
                                 ('V12 2021-2024', RUNS_MY, 'V12_window3_balanced_stats', f'_y{y}'),
                                 ('V16 2021-2024 (boundary)', RUNS_MY, 'V16_window3_boundary', f'_y{y}')]:
        d = f'{runs}/eval_{var}{tag}'
        try:
            m = json.load(open(f'{d}/metrics.json')) if glob.glob(f'{d}/metrics.json') else {}
            b = pd.read_csv(f'{d}/boundary.csv'); c = pd.read_csv(f'{d}/per_city.csv')
            rows.append({'year': y, 'model': name, 'mIoU_dw (mean of regions)': round(c.dw_miou.mean(), 3),
                         'OA': round(c.dw_oa.mean(), 3), 'boundary F1 raw': round(b.bf1_raw.mean(), 3)})
        except Exception as e:
            rows.append({'year': y, 'model': name, 'mIoU_dw (mean of regions)': f'missing: {type(e).__name__}'})
pd.DataFrame(rows)

## Decision
Ship the new model only if, on 2025 **and** 2026 held-out regions, it is better than the shipped 2021-only V12 on mIoU, and cities
are not worse. Ship V16 over V12 only if boundary F1 rises and mIoU does not fall by more than 0.01 (seed noise is about 0.01-0.02).
If neither clears that, keep the shipped model and report the result: it is still a held-out-in-time test of the shipped model.

To package the winner (replace `RUNS_MY`/variant/tag with the winner):
`!python scripts/package_model.py --runs $RUNS_MY --variant V16_window3_boundary --eval-tag _y2025`